# Wikipedia Text Extraction for Defense Entities

This notebook extracts Wikipedia article content for defense-domain entities identified in the Wikidata extraction pipeline. It performs memory-efficient batch processing with multi-threaded requests.

### Pipeline Steps
1. Read entity CSV with Wikipedia flags from the Wikidata extraction stage
2. Fetch Wikipedia summaries, full text, sections, categories, and links
3. Save results incrementally in chunks for fault tolerance
4. Generate enriched CSV and extraction report

### Requirements
```
pip install wikipedia-api pandas tqdm
```

### Input
- `defence_entities.csv` — Output from the Wikidata extraction pipeline

## Imports

In [ ]:
"""
Optimized Wikipedia Text Extractor from CSV
Memory-efficient batch processing with progress tracking
"""

import pandas as pd
import wikipediaapi
import json
import time
from typing import Dict, List, Optional
from tqdm import tqdm
from pathlib import Path
import logging
from concurrent.futures import ThreadPoolExecutor, as_completed
from collections import deque

In [ ]:
# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)


## Wikipedia Text Extractor

In [ ]:
class OptimizedWikipediaExtractor:
    """Memory-efficient Wikipedia text extractor with chunked processing"""

    def __init__(self, language='en', max_workers=5):
        """
        Initialize the extractor

        Args:
            language: Wikipedia language edition
            max_workers: Number of concurrent threads (recommended: 3-5)
        """
        self.wiki = wikipediaapi.Wikipedia(
            language=language,
            user_agent='DefenceDataPipeline/2.0 (Educational Project; batch processing)'
        )
        self.max_workers = max_workers
        self.cache = {}
        self.failed_entities = []

    def extract_page_content(self, title: str, entity_id: str = None) -> Optional[Dict]:
        """
        Extract Wikipedia page content with error handling

        Args:
            title: Wikipedia page title
            entity_id: Wikidata entity ID (for tracking)

        Returns:
            Dictionary with page content or None if failed
        """
        # Check cache first
        if title in self.cache:
            return self.cache[title]

        try:
            page = self.wiki.page(title)

            if not page.exists():
                logger.debug(f"Page not found: {title}")
                return None

            # Extract structured content
            content = {
                "entity_id": entity_id,
                "title": page.title,
                "summary": page.summary[:3000],  # First 2000 chars
                "full_text": page.text[:10000],  # First 10000 chars
                "url": page.fullurl,
                "categories": list(page.categories.keys())[:30],
                "links": list(page.links.keys())[:100],
                "sections": self._extract_sections(page),
                "text_length": len(page.text)
            }

            # Cache the result
            self.cache[title] = content
            return content

        except Exception as e:
            logger.warning(f"Failed to fetch '{title}': {str(e)}")
            self.failed_entities.append({
                "entity_id": entity_id,
                "title": title,
                "error": str(e)
            })
            return None

    def _extract_sections(self, page) -> List[Dict]:
        """Extract structured sections from Wikipedia page"""
        sections = []
        try:
            for i, section in enumerate(page.sections):
                if i >= 15:  # Limit to first 15 sections
                    break
                sections.append({
                    "title": section.title,
                    "text": section.text[:1000],  # First 1000 chars per section
                    "level": section.level
                })
        except Exception as e:
            logger.debug(f"Error extracting sections: {e}")
        return sections

    def process_single_entity(self, row: pd.Series) -> Optional[Dict]:
        """Process a single entity row"""
        # Skip if already has Wikipedia or no URL
        if not row.get('has_wikipedia', False):
            return None

        # Try multiple title sources
        title = None
        if pd.notna(row.get('wikipedia_title')):
            title = row['wikipedia_title']
        elif pd.notna(row.get('entity_label')):
            title = row['entity_label']

        if not title:
            return None

        return self.extract_page_content(
            title=title,
            entity_id=row.get('entity_id')
        )

    def process_csv_chunked(
        self,
        csv_path: str,
        output_dir: str = "./wikipedia_output",
        chunk_size: int = 100,
        delay_per_request: float = 0.15,
        use_threading: bool = True
    ):
        """
        Process CSV in memory-efficient chunks

        Args:
            csv_path: Path to input CSV
            output_dir: Output directory for results
            chunk_size: Number of rows per chunk (reduce for low memory)
            delay_per_request: Delay between requests (seconds)
            use_threading: Enable multi-threading (faster but more load)
        """
        output_path = Path(output_dir)
        output_path.mkdir(parents=True, exist_ok=True)

        # Read CSV metadata
        logger.info(f"Reading CSV: {csv_path}")
        df = pd.read_csv(csv_path)
        total_rows = len(df)

        # Filter rows with Wikipedia
        df_with_wiki = df[df['has_wikipedia'] == True].copy()
        total_to_process = len(df_with_wiki)

        logger.info(f"Total entities: {total_rows}")
        logger.info(f"Entities with Wikipedia: {total_to_process}")
        logger.info(f"Processing in chunks of {chunk_size}")

        all_results = []
        processed_count = 0
        chunk_num = 0

        # Process in chunks
        for chunk_start in range(0, total_to_process, chunk_size):
            chunk_end = min(chunk_start + chunk_size, total_to_process)
            chunk = df_with_wiki.iloc[chunk_start:chunk_end]
            chunk_num += 1

            logger.info(f"\n{'='*60}")
            logger.info(f"Processing Chunk {chunk_num} ({chunk_start+1}-{chunk_end}/{total_to_process})")
            logger.info(f"{'='*60}")

            chunk_results = []

            if use_threading:
                # Multi-threaded processing
                with ThreadPoolExecutor(max_workers=self.max_workers) as executor:
                    futures = {
                        executor.submit(self.process_single_entity, row): idx
                        for idx, row in chunk.iterrows()
                    }

                    for future in tqdm(
                        as_completed(futures),
                        total=len(futures),
                        desc=f"Chunk {chunk_num}",
                        unit="entity"
                    ):
                        result = future.result()
                        if result:
                            chunk_results.append(result)
                        time.sleep(delay_per_request)
            else:
                # Single-threaded processing (more conservative)
                for idx, row in tqdm(
                    chunk.iterrows(),
                    total=len(chunk),
                    desc=f"Chunk {chunk_num}",
                    unit="entity"
                ):
                    result = self.process_single_entity(row)
                    if result:
                        chunk_results.append(result)
                    time.sleep(delay_per_request)

            processed_count += len(chunk_results)
            all_results.extend(chunk_results)

            # Save chunk results incrementally
            chunk_file = output_path / f"wikipedia_chunk_{chunk_num}.json"
            with open(chunk_file, 'w', encoding='utf-8') as f:
                json.dump(chunk_results, f, indent=2, ensure_ascii=False)

            logger.info(f"✓ Chunk {chunk_num} complete: {len(chunk_results)} entities extracted")
            logger.info(f"  Saved to: {chunk_file}")

            # Progress summary
            success_rate = (processed_count / (chunk_end - 0)) * 100
            logger.info(f"  Overall progress: {processed_count}/{total_to_process} ({success_rate:.1f}%)")

            # Clear cache periodically to save memory
            if chunk_num % 5 == 0:
                cache_size = len(self.cache)
                self.cache.clear()
                logger.info(f"  Cache cleared ({cache_size} entries)")

        # Save consolidated results
        logger.info(f"\n{'='*60}")
        logger.info("Consolidating results...")
        logger.info(f"{'='*60}")

        consolidated_file = output_path / "wikipedia_all_entities.json"
        with open(consolidated_file, 'w', encoding='utf-8') as f:
            json.dump(all_results, f, indent=2, ensure_ascii=False)

        # Create enriched CSV
        self._create_enriched_csv(df, all_results, output_path)

        # Save failed entities log
        if self.failed_entities:
            failed_file = output_path / "failed_entities.json"
            with open(failed_file, 'w', encoding='utf-8') as f:
                json.dump(self.failed_entities, f, indent=2, ensure_ascii=False)
            logger.warning(f"Failed entities: {len(self.failed_entities)} (see {failed_file})")

        # Generate summary report
        self._generate_report(df, all_results, output_path)

        logger.info(f"\n{'='*60}")
        logger.info("EXTRACTION COMPLETE")
        logger.info(f"{'='*60}")
        logger.info(f"Total processed: {processed_count}/{total_to_process}")
        logger.info(f"Success rate: {(processed_count/total_to_process)*100:.1f}%")
        logger.info(f"Output directory: {output_path}")

        return all_results

    def _create_enriched_csv(self, original_df: pd.DataFrame, results: List[Dict], output_path: Path):
        """Create enriched CSV with Wikipedia data"""
        # Create a mapping of entity_id to Wikipedia data
        wiki_data_map = {r['entity_id']: r for r in results}

        # Add Wikipedia columns
        enriched_df = original_df.copy()
        enriched_df['wikipedia_extracted'] = enriched_df['entity_id'].apply(
            lambda x: x in wiki_data_map
        )
        enriched_df['wikipedia_summary_length'] = enriched_df['entity_id'].apply(
            lambda x: len(wiki_data_map[x]['summary']) if x in wiki_data_map else 0
        )
        enriched_df['wikipedia_text_length'] = enriched_df['entity_id'].apply(
            lambda x: wiki_data_map[x].get('text_length', 0) if x in wiki_data_map else 0
        )
        enriched_df['wikipedia_num_sections'] = enriched_df['entity_id'].apply(
            lambda x: len(wiki_data_map[x].get('sections', [])) if x in wiki_data_map else 0
        )

        csv_file = output_path / "defence_entities_enriched.csv"
        enriched_df.to_csv(csv_file, index=False, encoding='utf-8')
        logger.info(f"✓ Enriched CSV saved: {csv_file}")

    def _generate_report(self, original_df: pd.DataFrame, results: List[Dict], output_path: Path):
        """Generate summary report"""
        total_entities = len(original_df)
        entities_with_wiki_flag = len(original_df[original_df['has_wikipedia'] == True])
        successfully_extracted = len(results)

        report = {
            "extraction_date": time.strftime("%Y-%m-%d %H:%M:%S"),
            "input_csv_stats": {
                "total_entities": total_entities,
                "entities_with_wikipedia_flag": entities_with_wiki_flag,
                "percentage_with_wikipedia": round(
                    (entities_with_wiki_flag / total_entities) * 100, 2
                )
            },
            "extraction_stats": {
                "successfully_extracted": successfully_extracted,
                "failed": len(self.failed_entities),
                "success_rate": round(
                    (successfully_extracted / entities_with_wiki_flag) * 100, 2
                ) if entities_with_wiki_flag > 0 else 0
            },
            "content_stats": {
                "total_text_extracted_chars": sum(r.get('text_length', 0) for r in results),
                "avg_text_length": round(
                    sum(r.get('text_length', 0) for r in results) / successfully_extracted, 2
                ) if successfully_extracted > 0 else 0,
                "avg_sections_per_page": round(
                    sum(len(r.get('sections', [])) for r in results) / successfully_extracted, 2
                ) if successfully_extracted > 0 else 0
            },
            "output_files": [
                "wikipedia_all_entities.json - Consolidated results",
                "defence_entities_enriched.csv - CSV with Wikipedia stats",
                "wikipedia_chunk_*.json - Individual chunk files",
                "extraction_report.json - This report"
            ]
        }

        report_file = output_path / "extraction_report.json"
        with open(report_file, 'w', encoding='utf-8') as f:
            json.dump(report, f, indent=2, ensure_ascii=False)

        # summary
        summary_file = output_path / "EXTRACTION_SUMMARY.txt"
        with open(summary_file, 'w', encoding='utf-8') as f:
            f.write("="*60 + "\n")
            f.write("WIKIPEDIA TEXT EXTRACTION SUMMARY\n")
            f.write("="*60 + "\n\n")
            f.write(f"Date: {report['extraction_date']}\n\n")
            f.write(f"Input Statistics:\n")
            f.write(f"  Total entities: {total_entities}\n")
            f.write(f"  Entities with Wikipedia: {entities_with_wiki_flag}\n\n")
            f.write(f"Extraction Results:\n")
            f.write(f"  Successfully extracted: {successfully_extracted}\n")
            f.write(f"  Failed: {len(self.failed_entities)}\n")
            f.write(f"  Success rate: {report['extraction_stats']['success_rate']}%\n\n")
            f.write(f"Content Statistics:\n")
            f.write(f"  Total text: {report['content_stats']['total_text_extracted_chars']:,} chars\n")
            f.write(f"  Avg text per page: {report['content_stats']['avg_text_length']:,.0f} chars\n")
            f.write(f"  Avg sections per page: {report['content_stats']['avg_sections_per_page']:.1f}\n")
            f.write("\n" + "="*60 + "\n")

        logger.info(f"✓ Report saved: {summary_file}")



## Run Extraction

In [ ]:
def main():
    """Main execution function with example usage"""

    print("\n🚀 Wikipedia Text Extractor from CSV")
    print("="*60)

    # Configuration
    CSV_PATH = "./defence_entities.csv"  # Your input CSV
    OUTPUT_DIR = "./wikipedia_extraction_output"

    # Initialize extractor
    extractor = OptimizedWikipediaExtractor(
        language='en',
        max_workers=4  # Adjust based on your needs (3-5 recommended)
    )

    # Process the CSV
    # Parameters to tune:
    # - chunk_size: Larger = faster but more memory (50-200 recommended)
    # - delay_per_request: Smaller = faster but more aggressive (0.05-0.2 recommended)
    # - use_threading: True for speed, False for conservative approach

    try:
        results = extractor.process_csv_chunked(
            csv_path=CSV_PATH,
            output_dir=OUTPUT_DIR,
            chunk_size=500,  # Process 100 entities at a time
            delay_per_request=0.15,  # 100ms delay between requests
            use_threading=True  # Enable multi-threading
        )

        print(f"\n✅ Extraction complete!")
        print(f"   Total pages extracted: {len(results)}")
        print(f"   Output location: {OUTPUT_DIR}")
        print(f"   Check '{OUTPUT_DIR}/EXTRACTION_SUMMARY.txt' for details")

    except FileNotFoundError:
        print(f"\n❌ Error: CSV file not found at '{CSV_PATH}'")
        print("   Please update CSV_PATH in the script")
    except Exception as e:
        print(f"\n❌ Error during extraction: {e}")
        logger.exception("Extraction failed")


In [ ]:
if __name__ == "__main__":
    main()